In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("budgetwise_finance_dataset.csv")

print("Raw shape:", df.shape) 
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

In [6]:
df = df.drop_duplicates()
df = df.drop(columns=["notes"])

In [7]:
df["date_clean"] = pd.to_datetime(df["date"], format="mixed", errors="coerce")
df = df.dropna(subset=["date_clean"])

In [8]:
def clean_amount(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    s = s.replace("Rs.", "").replace("INR", "")
    s = s.replace("$", "").replace("₹", "")
    s = s.strip()
    try: return float(s)
    except ValueError: return np.nan

df["amount_clean"] = df["amount"].apply(clean_amount)
df = df.dropna(subset=["amount_clean"])

df["is_refund"] = df["amount_clean"] < 0
df["amount_clean"] = df["amount_clean"].abs()

In [ ]:
category_map = {"fod": "Food", "foodd": "Food", "foods": "Food",
    "educaton": "Education", "edu": "Education",
    "entrtnmnt": "Entertainment", "entertain": "Entertainment",
    "helth": "Health",
    "rentt": "Rent", "rnt": "Rent",
    "travl": "Travel", "traval": "Travel",
    "utilties": "Utilities", "utlities": "Utilities",
    "utility": "Utilities",
    "other": "Others",
    "saving": "Savings",
}

df["category_clean"] = (df["category"].str.strip().str.lower().replace(category_map).str.title())
df["category_clean"] = df["category_clean"].fillna("Uncategorized")

print("Categories:", df["category_clean"].nunique())

In [ ]:
category_map = {"fod": "Food", "foodd": "Food", "foods": "Food",
    "educaton": "Education", "edu": "Education",
    "entrtnmnt": "Entertainment", "entertain": "Entertainment",
    "helth": "Health",
    "rentt": "Rent", "rnt": "Rent",
    "travl": "Travel", "traval": "Travel",
    "utilties": "Utilities", "utlities": "Utilities",
    "utility": "Utilities",
    "other": "Others",
    "saving": "Savings",
}

df["category_clean"] = (df["category"].str.strip().str.lower().replace(category_map).str.title())
df["category_clean"] = df["category_clean"].fillna("Uncategorized")

print("Categories:", df["category_clean"].nunique())

In [ ]:
city_map = {"ahm": "Ahmedabad", "ban": "Bangalore",
    "che": "Chennai", "del": "Delhi",
    "hyd": "Hyderabad", "jai": "Jaipur",
    "kol": "Kolkata", "luc": "Lucknow",
    "mum": "Mumbai", "pun": "Pune",
}
df["location_clean"] = (df["location"].str.strip().str.lower().replace(city_map).str.title())
df["location_clean"] = df.groupby("user_id")["location_clean"].transform(lambda x: x.fillna(x.mode()[0]
if not x.mode().empty 
else "Unknown"))

print("Cities:", df["location_clean"].nunique())

In [ ]:
Q1 = df["amount_clean"].quantile(0.25)
Q3 = df["amount_clean"].quantile(0.75)
IQR = Q3 - Q1
df["is_outlier"] = df["amount_clean"] > (Q3 + 1.5 * IQR)

print("Outliers flagged:", df["is_outlier"].sum())

df.to_csv("upi_clean.csv", index=False)

print("Final clean rows:", len(df))